# NAOMI encoder + decoder training (Colab)

Trains **both** halves of the autoencoder in one run:

1. `nsm_ct.encoder_model.EncoderModel` (the candidate-lattice encoder, `dev/ENCODER_MODEL_SPEC.md`) on `runs/encoder_gold_v2.jsonl`.
2. `nsm_ct.decoder_trained.DecoderTrainedModel` (the learned reconstruction decoder, RESEARCH_NOTES.md "DECODER PLAN UPDATE") on the same corpus, self-supervised by reconstruction.

Then it chains the two together for the metric the lead asked for -- the **round-trip** autoencoder test:

```
text -> encoder.beam_decode -> top committed tree -> decoder.realize -> text
```

and reports, in one final block:

- English candidate-set recall (sense / slot / structure) vs a random-legal baseline, on a held-out split.
- The Spanish grammar-swap eval (English-trained encoder weights, zero Spanish training) vs random.
- Decoder reconstruction (exact-match + token-F1) fed the **gold** committed tree, on held-out records.
- The **round-trip** exact-match + token-F1: encoder and decoder working together, no gold tree anywhere in that path.
- A no-confab spot check: a severed (content-stripped) structure must make the decoder abstain (empty output), never confabulate.

Run the three code cells below in order. Everything (repo clone, deps, USVS build, gold-data fetch, training, eval) happens inside this notebook -- no local setup needed.

In [ ]:
!git clone -b colab-full-notebook https://github.com/LinguisticsDevelopment/NAOMI.git && cd NAOMI/consciousness_transformer && pip install -q torch numpy nltk pytest && pip install -e .

In [ ]:
%cd NAOMI/consciousness_transformer
import os
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"
!python -c "import nltk; nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True); nltk.download('omw-2.0', quiet=True)"
!python scripts/build_usvs.py
!mkdir -p runs
!git show origin/encoder-gold-v2:consciousness_transformer/runs/encoder_gold_v2.jsonl > runs/encoder_gold_v2.jsonl
!git show origin/spanish-gold-v2:consciousness_transformer/runs/spanish_gold_v2.jsonl > runs/spanish_gold_v2.jsonl
!wc -l runs/encoder_gold_v2.jsonl runs/spanish_gold_v2.jsonl

In [ ]:
!python scripts/colab_train_all.py --enc-records 984 --enc-epochs 50 --dec-records 984 --dec-epochs 80 --device cpu --outdir runs/colab_all

## Reading the FINAL RESULTS block

The last cell prints a `FINAL RESULTS` block with four sections:

- **ENCODER** -- English (held-out test split) `model` vs `random` candidate-set recall for sense / slot / structure, then the Spanish grammar-swap eval on the SAME English-trained checkpoint (`model` vs `random`) -- the interlingua claim: one frozen encoder, a swapped-language front end, no Spanish training data.
- **DECODER** -- reconstruction exact-match + token-F1 on held-out records, fed each record's own **gold** committed tree (no encoder involved yet -- this isolates decoder quality).
- **ROUND-TRIP** -- the encoder+decoder autoencoder metric: `text -> encoder.beam_decode -> top tree -> decoder.realize -> text`, exact-match + token-F1 on held-out English sentences never used to train the decoder. This is the number that matters for "does the whole pipeline work end to end."
- **NO-CONFAB SPOT CHECK** -- severs the committed structure's content (keeps shape, nulls every surface word) and confirms `realize()` returns nothing; this is a structural gate in `nsm_ct/decoder_trained.py`, not a trained behaviour, so it should always read `all_abstained=True`.

**Device / timing:** the encoder is CPU-only as of this snapshot of `nsm_ct/encoder_model.py` (its controller builds plain CPU index tensors for every embedding lookup, so `.to('cuda')` is a no-op the driver detects and reports rather than erroring on) -- a GPU runtime will not speed up the encoder phase; the decoder is tiny (a 48-d GRU) and CPU-fast regardless. Use a CPU runtime, no need to burn a GPU allocation.

**Defaults and wall-clock:** `--enc-records 984 --enc-epochs 50` reproduces the spec's full-Stage-i split (788/98/98 of the 985 available English gold records) and, from this repo's own CPU smoke timings, lands around 45-60 minutes for the encoder phase alone. `--dec-records 984 --dec-epochs 80` trains the decoder on ~787 records at ~15-17s/epoch (~20-22 minutes). Together with the Spanish/English eval passes and the round-trip + no-confab checks (a few minutes), the **total run should land at roughly 70-95 minutes**, comfortably inside a single ~2h Colab session. Lower `--enc-records`/`--enc-epochs`/`--dec-records`/`--dec-epochs` for a faster, noisier run.

### Getting the checkpoints out

Two checkpoints are saved under `runs/colab_all/` (relative to `NAOMI/consciousness_transformer`, i.e. `/content/NAOMI/consciousness_transformer/runs/colab_all/` in a fresh Colab runtime): `encoder_colab.pt` and `decoder_colab.pt`. Either:

```python
from google.colab import files
files.download('runs/colab_all/encoder_colab.pt')
files.download('runs/colab_all/decoder_colab.pt')
```

or mount Drive and copy them there before the runtime recycles:

```python
from google.colab import drive
drive.mount('/content/drive')
!cp runs/colab_all/encoder_colab.pt runs/colab_all/decoder_colab.pt /content/drive/MyDrive/
```